# Fink/LSST — Statistics of Alerts on Heatmap

This notebook reads the Parquet files saved by `01_fink_block_lightcurves.ipynb`  
and reproduces the same visualisations (flux and magnitude) per source group.

**Expected data** in `data_FINK_BLOCK_LC_01/`:
- `{group}_fp.parquet`  — forced-photometry light curves
- `{group}_src.parquet` — detection-based light curves (diaSources)
- `flatness_metrics.csv` — pre-computed flatness metrics

No API call is made in this notebook.

- **author** : Sylvie Dagoret-Campane
- **affliliation** : IJCLab/CNRS
- **creation date** : 2026-04-02
- **last update** : 2026-04-03
- **subject:** Fink alert Broker applied to Rubin LSST alertes

## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm
from scipy.optimize import curve_fit
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01"
DIR_DATA = f"data_{NB_TAG}"  # input: written by notebook 01
DIR_FIGS = f"figs_{NB_TAG}_05"  # output: figures specific to this notebook
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Display parameters ────────────────────────────────────────────────────────
NC = 20  # maximum number of light curves to plot per group
BANDS = list("ugrizy")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)

AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


# ── Plotting mode ────────────────────────────────────────────────────────
# Choose which groups to display:
#   'all'     -> all groups found on disk (default)
#   'calib'   -> only groups usable for photometric calibration
#   'exclude' -> only groups EXCLUDED from calibration
#                (variable stars, SSO objects, TNS transients, blazars...)
PLOT_MODE = "all"  # <── change here: 'all' | 'calib' | 'exclude'

# Groups excluded from calibration (consistent with notebook 01).
# Any group whose name STARTS WITH 'simbad_variable' is also excluded.
CALIBRATION_EXCLUDE = {
    "solar_system",
    "tns_transient",
    "gaia_star_variable",
    "vsx_variable",
    "gcvs_variable",
    "spicy_yso",
    "blazar_x3hsp",
    "blazar_4lac",
}

print(f"Data directory   : {os.path.abspath(DIR_DATA)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Plot mode        : {PLOT_MODE!r}")

## 2. Utility functions (identical to notebook 01 + luptitude support)

In [ ]:
def flux_to_mag(flux_nJy, flux_err_nJy=None):
    """Convert PSF flux (nJy) to AB magnitude, propagating uncertainties.

    Negative or zero flux values are silently returned as NaN.

    Parameters
    ----------
    flux_nJy : array-like
        PSF flux in nano-Janskys.
    flux_err_nJy : array-like or None
        1-sigma flux uncertainty in nJy.  If None, only magnitudes are returned.

    Returns
    -------
    mag : ndarray
        AB magnitude (NaN where flux <= 0).
    mag_err : ndarray or None
        Magnitude uncertainty (NaN where flux <= 0), or None if not requested.
    """
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    """Normalised RMS variability: sigma(<f>) / <f>, computed on finite positive flux values."""
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


print("Utility functions defined.")

In [ ]:
def flux_to_luptitude(flux_nJy, flux_err_nJy, zero_point=31.4):
    """Convert PSF flux (nJy) to asinh magnitudes (Luptitudes).

    Luptitudes (Lupton et al. 1999) handle zero and negative flux values
    that arise in difference-image photometry (DIA).  They behave as
    standard AB magnitudes at high signal-to-noise and transition to a
    linear flux scale near the noise floor, so every measurement is
    plotted -- including those with negative flux.

    The softening parameter *b* is set to the median flux uncertainty of
    the input array, placing the linear/log transition at the 1-sigma
    noise level.  This matches the convention used in the DP2 DiaObject
    Butler notebook (``plot_dia_object_luptitudes``).

    Formula
    -------
    mu = -2.5 * log10(e) * [arcsinh(f / 2b) + ln(b)] + (zp + 2.5*log10(b))

    Error propagation
    -----------------
    sigma_mu = 1.085736 * sigma_f / sqrt(f^2 + (2b)^2)

    Parameters
    ----------
    flux_nJy : array-like
        PSF flux in nano-Janskys.  May contain negative values.
    flux_err_nJy : array-like
        1-sigma flux uncertainty in nJy (used to derive the softening
        parameter b = median(flux_err_nJy)).
    zero_point : float, optional
        AB zero-point offset in nJy units (default 31.4 for nJy).

    Returns
    -------
    lup : ndarray
        Asinh magnitude (Luptitude) for each measurement.
    lup_err : ndarray
        1-sigma uncertainty on the Luptitude.
    b : float
        Softening parameter used (= median of flux_err_nJy).
    """
    flux = np.asarray(flux_nJy, dtype=float)
    err = np.asarray(flux_err_nJy, dtype=float)

    # Softening parameter: median noise level of the input data
    b = float(np.nanmedian(err))
    if not np.isfinite(b) or b <= 0.0:
        b = 1.0  # safe fallback

    # Pre-factor  2.5 / ln(10) ~ 1.085736
    log10_e_inv = 1.085736

    # Asinh magnitude (Luptitude)
    lup = -2.5 * log10_e_inv * (np.arcsinh(flux / (2.0 * b)) + np.log(b)) + (zero_point + 2.5 * np.log10(b))

    # Error propagation
    lup_err = log10_e_inv * err / np.sqrt(flux**2 + (2.0 * b) ** 2)

    return lup, lup_err, b


print("flux_to_luptitude() defined.")

In [ ]:
def plot_focal_plane_heatmap_vectorized(
    alerts_df,
    geometry_csv,
    value_col="r:visit",
    agg_func="count",
    cmap="viridis",
    log_scale=True,
    figsize=(8, 8),
    show_colorbar=True,
    title="Focal Plane Heatmap",
):

    # --- Load geometry
    geom = pd.read_csv(geometry_csv)

    # --- Compute values per detector
    values = alerts_df.groupby("r:detector")[value_col].agg(agg_func)
    geom["value"] = geom["detector"].map(values).fillna(0).astype(float)

    # Must compute boundaries
    xmin = geom[[f"corner{i}_x" for i in range(4)]].min().min()
    xmax = geom[[f"corner{i}_x" for i in range(4)]].max().max()
    ymin = geom[[f"corner{i}_y" for i in range(4)]].min().min()
    ymax = geom[[f"corner{i}_y" for i in range(4)]].max().max()

    # --- Handle log scale and normalization
    if log_scale:
        vmin = max(geom["value"][geom["value"] > 0].min(), 1e-2)
        vmax = geom["value"].max()
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    else:
        norm = mcolors.Normalize(vmin=geom["value"].min(), vmax=geom["value"].max())

    cmap = plt.get_cmap(cmap)

    # --- Plot
    fig, ax = plt.subplots(figsize=figsize)

    # Vectorisé : on crée toutes les patches en une fois
    for i, row in geom.iterrows():
        corners = [(row[f"corner{j}_x"], row[f"corner{j}_y"]) for j in range(4)]
        val = row["value"]
        color = cmap(norm(val)) if val > 0 else "lightgrey"
        poly = patches.Polygon(corners, facecolor=color, edgecolor="black", linewidth=0.2)
        ax.add_patch(poly)
        ax.text(
            row["x_center"],
            row["y_center"],
            f"{int(row['detector'])}\n{row['name']}",
            ha="center",
            va="center",
            fontsize=6,
            fontweight="bold",
        )

    ax.set_aspect("equal")
    # ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Focal plane X")
    ax.set_ylabel("Focal plane Y")
    ax.set_title(title)

    # --- Colorbar
    if show_colorbar:
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        # sm.set_array([])
        plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

## 3. Load Parquet files from disk

In [ ]:
# Rubin LSST Focal Plane Geometry

In [ ]:
PATH_FOCALPLANEGEOM_filename = "data_FocalPlane/ccd_geometry.csv"
geometry_csv = PATH_FOCALPLANEGEOM_filename

In [ ]:
# Auto-discover available groups from file names
fp_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_fp.parquet")))
src_files = sorted(glob.glob(os.path.join(DIR_DATA, "*_src.parquet")))


def group_from_path(path, suffix):
    return os.path.basename(path).replace(suffix, "")


groups_fp = {group_from_path(p, "_fp.parquet"): p for p in fp_files}
groups_src = {group_from_path(p, "_src.parquet"): p for p in src_files}
all_groups = sorted(set(groups_fp) | set(groups_src))


# ── Apply PLOT_MODE filter ───────────────────────────────────────────────
def _is_excluded(g):
    return g in CALIBRATION_EXCLUDE or g.startswith("simbad_variable")


if PLOT_MODE == "calib":
    selected_groups = [g for g in all_groups if not _is_excluded(g)]
elif PLOT_MODE == "exclude":
    selected_groups = [g for g in all_groups if _is_excluded(g)]
else:  # 'all'
    selected_groups = list(all_groups)

print(f"Groups on disk: {len(all_groups)},  selected (PLOT_MODE={PLOT_MODE!r}): {len(selected_groups)}")
for g in all_groups:
    has_fp = "yes" if g in groups_fp else "no"
    has_src = "yes" if g in groups_src else "no"
    sel = "<<" if g in selected_groups else "  "
    label = "[EXCL] " if _is_excluded(g) else "[calib]"
    print(f"  {sel} {label} {g:40s}  fp={has_fp:3s}  src={has_src}")

In [ ]:
# Load all Parquet files and reconstruct the lc_cache dictionary.
# Structure: lc_cache[group][oid] = {'fp': DataFrame, 'src': DataFrame}

lc_cache = {}

for group in all_groups:
    lc_cache[group] = {}

    for tag, path_dict in (("fp", groups_fp), ("src", groups_src)):
        if group not in path_dict:
            continue
        df = pd.read_parquet(path_dict[group])

        # Drop any residual NaN rows in core columns
        df = df.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)

        # Recompute mag / mag_err if absent (e.g. columns were not saved)
        if "mag" not in df.columns or "mag_err" not in df.columns:
            mag, mag_err = flux_to_mag(df["r:psfFlux"].values, df["r:psfFluxErr"].values)
            df["mag"] = mag
            df["mag_err"] = mag_err
            df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)

        # Split by object
        for oid, sub in df.groupby("diaObjectId"):
            oid = str(oid)
            if oid not in lc_cache[group]:
                lc_cache[group][oid] = {"fp": pd.DataFrame(), "src": pd.DataFrame()}
            lc_cache[group][oid][tag] = sub.reset_index(drop=True)

# Summary
print("Objects loaded per group:")
for g in all_groups:
    n_oids = len(lc_cache[g])
    n_pts = sum(len(d["fp"]) + len(d["src"]) for d in lc_cache[g].values())
    print(f"  {g:35s}  {n_oids:3d} objects  {n_pts:6d} data points")

In [ ]:
# lc_cache['gaia_star_stable']['170028485670076523']['fp']

In [ ]:
# lc_cache['gaia_star_stable']['170028485670076523']['src']

## Prepare heatmap

In [ ]:
def concat_group_src(lc_cache, group="gaia_star_stable"):
    dfs = []

    for diaObjectId, content in lc_cache[group].items():
        if "src" in content and content["src"] is not None:
            df = content["src"].copy()

            # garder la trace de l'origine (très important)
            df["diaObjectId"] = diaObjectId

            dfs.append(df)

    if len(dfs) == 0:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)

In [ ]:
df_alerts_gaia_star_stable = concat_group_src(lc_cache, "gaia_star_stable")
df_alerts_gaia_star_stable_hq = concat_group_src(lc_cache, "gaia_star_stable_hq")

In [ ]:
df_all_alerts_gaia_star = pd.concat([df_alerts_gaia_star_stable, df_alerts_gaia_star_stable_hq])

In [ ]:
df_all_alerts_gaia_star.groupby("r:detector")["r:visit"].count().describe()

## Plot Focal Plane Heatmap

In [ ]:
plot_focal_plane_heatmap_vectorized(
    df_all_alerts_gaia_star,
    geometry_csv,
    value_col="r:visit",  # None = counts, sinon colonne à agréger
    agg_func="count",  # "count", "mean", etc.
    cmap="viridis",
    log_scale=True,
    figsize=(8, 8),
    show_colorbar=True,
    title="Alert counts on LSSTCam Focal Plane Heatmap",
)

## Radial projection: alert count vs. distance from camera centre

Each CCD is projected onto a single radial coordinate
$r = \sqrt{x_{\rm center}^2 + y_{\rm center}^2}$ [mm],
where $(x_{\rm center},\, y_{\rm center})$ are the CCD centre coordinates
in the focal-plane frame (origin = optical axis of the camera).

### Physical model

The effective collecting area of a CCD tilted by an angle $\theta$ with respect to
the principal axis of the camera falls off as $\cos\theta$.  For a wide-field
telescope the off-axis angle is related to the focal-plane radius by
$\theta \approx r / f$, where $f$ is the effective focal length.  We therefore
fit the alert count per CCD with:

$$N(r) = A \cos(B \cdot r)$$

where $A$ is the on-axis normalisation and $B = 1/f_{\rm eff}$ encodes the
focal-length scale.  The argument $B \cdot r$ is in radians.

### Fit range

The fit is restricted to CCDs with $r \leq R_{\rm fit}$ (default 100 mm).
CCDs beyond this radius are shown on the plot as open markers but **excluded
from the fit**.  The analytical model is then **extrapolated** beyond
$R_{\rm fit}$ using a dash-dot line, which allows a direct visual comparison
between the fitted cosine and the outer-field data.

### Uncertainties

Alert counts follow approximately Poisson statistics, so the 1-sigma uncertainty
on each CCD count is $\sigma_N = \sqrt{N}$.  These are used as weights in the
weighted least-squares fit (`sigma` parameter of `scipy.optimize.curve_fit`).

In [ ]:
# ── Build the per-CCD alert count table with radial distance ─────────────────

# Load geometry
geom = pd.read_csv(geometry_csv)

# Compute radial distance of each CCD centre from the camera optical axis
geom["r_mm"] = np.sqrt(geom["x_center"] ** 2 + geom["y_center"] ** 2)

# Count alerts per detector (same aggregation as the heatmap)
alert_counts_per_ccd = (
    df_all_alerts_gaia_star.groupby("r:detector")["r:visit"]
    .count()
    .rename("n_alerts")
    .reset_index()
    .rename(columns={"r:detector": "detector"})
)

# Merge geometry with alert counts
# CCDs with zero alerts are kept (n_alerts = 0) for completeness
geom_counts = geom.merge(alert_counts_per_ccd, on="detector", how="left")
geom_counts["n_alerts"] = geom_counts["n_alerts"].fillna(0).astype(float)

# Poisson error: sqrt(N); set to 1 for CCDs with 0 alerts to avoid division by zero
geom_counts["n_err"] = np.where(
    geom_counts["n_alerts"] > 0,
    np.sqrt(geom_counts["n_alerts"]),
    1.0,
)

# Keep only CCDs with at least one alert
geom_fit_all = geom_counts[geom_counts["n_alerts"] > 0].copy()

print(f"Total CCDs in geometry : {len(geom_counts)}")
print(f"CCDs with alerts       : {len(geom_fit_all)}")
print(f"r_mm range             : [{geom_fit_all['r_mm'].min():.1f},  {geom_fit_all['r_mm'].max():.1f}] mm")
print(
    f"N_alerts range         : [{geom_fit_all['n_alerts'].min():.0f},  {geom_fit_all['n_alerts'].max():.0f}]"
)

In [ ]:
# ── Fit N(r) = A * cos(B * r) — restricted to r <= R_FIT_MAX ─────────────────

# Maximum radius included in the fit.
# CCDs beyond R_FIT_MAX are displayed on the plot as open markers
# and described by the analytical extrapolation (dash-dot line).
R_FIT_MAX = 100.0  # mm  <── change here if needed


def cosine_model(r, A, B):
    """
    Cosine projection model for alert count vs. radial distance.

    N(r) = A * cos(B * r)

    Parameters
    ----------
    r : array-like  — radial distance from camera centre [mm]
    A : float       — on-axis normalisation (count at r=0)
    B : float       — angular scale factor [rad/mm] = 1 / f_eff

    Returns
    -------
    N : ndarray — predicted alert count
    """
    return A * np.cos(B * np.asarray(r, dtype=float))


# Split into fit range (inner) and display-only range (outer)
mask_inner = geom_fit_all["r_mm"] <= R_FIT_MAX
mask_outer = geom_fit_all["r_mm"] > R_FIT_MAX

geom_inner = geom_fit_all[mask_inner]  # used for the fit
geom_outer = geom_fit_all[mask_outer]  # shown but excluded from fit

r_data = geom_inner["r_mm"].values
N_data = geom_inner["n_alerts"].values
N_err = geom_inner["n_err"].values

print(f"Fit range  : r <= {R_FIT_MAX:.0f} mm  ->  {len(geom_inner)} CCDs used for fit")
print(f"Outer range: r >  {R_FIT_MAX:.0f} mm  ->  {len(geom_outer)} CCDs shown (extrapolation only)")

# Initial parameter guess:
#   A ~ maximum alert count in the fit range
#   B ~ 1/300 rad/mm  (typical effective focal length ~ 300 mm for Rubin)
p0 = [N_data.max(), 1.0 / 300.0]

# Bounds: A > 0, B > 0 (cosine must be decreasing outward)
bounds = ([0, 0], [np.inf, np.inf])

try:
    popt, pcov = curve_fit(
        cosine_model,
        r_data,
        N_data,
        p0=p0,
        sigma=N_err,
        absolute_sigma=True,
        bounds=bounds,
        maxfev=10000,
    )
    perr = np.sqrt(np.diag(pcov))  # 1-sigma parameter uncertainties
    A_fit, B_fit = popt
    A_err, B_err = perr

    # Effective focal length derived from the fit
    f_eff = 1.0 / B_fit if B_fit > 0 else np.nan
    f_eff_err = f_eff**2 * B_err if B_fit > 0 else np.nan

    # Reduced chi-squared — evaluated on the fit range only
    residuals = N_data - cosine_model(r_data, *popt)
    chi2_red = float(np.sum((residuals / N_err) ** 2) / (len(N_data) - 2))

    print(f"\nFit results  N(r) = A * cos(B * r)  [fit range r <= {R_FIT_MAX:.0f} mm]:")
    print(f"  A = {A_fit:.1f} +/- {A_err:.1f}  alerts")
    print(f"  B = {B_fit:.6f} +/- {B_err:.6f}  rad/mm")
    print(f"  Effective focal length  f_eff = 1/B = {f_eff:.1f} +/- {f_eff_err:.1f}  mm")
    print(f"  Reduced chi2 = {chi2_red:.3f}  (over {len(N_data)} CCDs in fit range)")
    fit_ok = True

except RuntimeError as exc:
    print(f"WARNING: curve_fit did not converge: {exc}")
    A_fit = B_fit = A_err = B_err = f_eff = f_eff_err = chi2_red = np.nan
    fit_ok = False

In [ ]:
# ── Error-bar plot N vs r  +  fit (solid, r<=R_FIT_MAX)  +  extrapolation (dash-dot) ──

r_max_plot = geom_fit_all["r_mm"].max() * 1.05  # x-axis upper limit

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

# ── Inner CCDs (fit range): filled circles, coloured by N ────────────────────
sc_in = ax.scatter(
    geom_inner["r_mm"],
    geom_inner["n_alerts"],
    c=geom_inner["n_alerts"],
    cmap="viridis",
    s=35,
    zorder=4,
    label=f"CCD  $r \\leq {R_FIT_MAX:.0f}$ mm  (fit range)",
)
ax.errorbar(
    geom_inner["r_mm"],
    geom_inner["n_alerts"],
    yerr=geom_inner["n_err"],
    fmt="none",
    ecolor="steelblue",
    elinewidth=0.8,
    capsize=2,
    alpha=0.6,
    zorder=3,
)

# ── Outer CCDs (excluded from fit): open circles, same colourmap ─────────────
if len(geom_outer) > 0:
    sc_out = ax.scatter(
        geom_outer["r_mm"],
        geom_outer["n_alerts"],
        c=geom_outer["n_alerts"],
        cmap="viridis",
        vmin=geom_fit_all["n_alerts"].min(),
        vmax=geom_fit_all["n_alerts"].max(),
        s=35,
        zorder=4,
        marker="o",
        facecolors="none",
        linewidths=1.2,
        label=f"CCD  $r > {R_FIT_MAX:.0f}$ mm  (excluded from fit)",
    )
    ax.errorbar(
        geom_outer["r_mm"],
        geom_outer["n_alerts"],
        yerr=geom_outer["n_err"],
        fmt="none",
        ecolor="steelblue",
        elinewidth=0.8,
        capsize=2,
        alpha=0.4,
        zorder=3,
    )

if fit_ok:
    # Dense r-grid covering the full plot range
    r_all = np.linspace(0, r_max_plot, 600)
    N_all = cosine_model(r_all, A_fit, B_fit)

    # Sub-arrays for the two line styles
    r_solid = r_all[r_all <= R_FIT_MAX]
    N_solid = cosine_model(r_solid, A_fit, B_fit)
    r_dash = r_all[r_all >= R_FIT_MAX]  # start exactly at boundary
    N_dash = cosine_model(r_dash, A_fit, B_fit)

    # Fit label (shown only once, on the solid segment)
    fit_label = (
        rf"Fit ($r \leq {R_FIT_MAX:.0f}$ mm): $N = A\,\cos(B\,r)$"
        f"\n$A = {A_fit:.0f} \\pm {A_err:.0f}$"
        f"\n$B = {B_fit:.5f} \\pm {B_err:.5f}$ rad/mm"
        f"\n$f_{{\\rm eff}} = 1/B = {f_eff:.0f} \\pm {f_eff_err:.0f}$ mm"
        f"\n$\\chi^2_{{\\rm red}} = {chi2_red:.2f}$"
    )

    # Solid line over the fit range
    ax.plot(r_solid, N_solid, color="tomato", lw=2.0, ls="-", label=fit_label, zorder=5)

    # Dash-dot line for the analytic extrapolation beyond R_FIT_MAX
    ax.plot(
        r_dash,
        N_dash,
        color="tomato",
        lw=1.8,
        ls="-.",
        label=rf"Analytic extrapolation ($r > {R_FIT_MAX:.0f}$ mm)",
        zorder=5,
    )

    # Vertical boundary line at R_FIT_MAX
    ax.axvline(R_FIT_MAX, color="gray", lw=1.0, ls=":", label=f"Fit boundary $r = {R_FIT_MAX:.0f}$ mm")

    # ±1-sigma uncertainty band over the full range
    N_upper = np.clip(cosine_model(r_all, A_fit + A_err, B_fit - B_err), 0, None)
    N_lower = np.clip(cosine_model(r_all, A_fit - A_err, B_fit + B_err), 0, None)
    ax.fill_between(
        r_all, N_lower, N_upper, color="tomato", alpha=0.12, label=r"Fit $\pm 1\sigma$ band", zorder=2
    )

# Colour bar (linked to the inner scatter; vmin/vmax shared with outer)
cbar = fig.colorbar(sc_in, ax=ax, fraction=0.03, pad=0.01)
cbar.set_label("N alerts per CCD", fontsize=8)

ax.set_xlabel(
    r"Radial distance from camera centre  $r = \sqrt{x^2 + y^2}$  [mm]",
    fontsize=10,
)
ax.set_ylabel("Number of alerts per CCD  $N$", fontsize=10)
ax.set_title(
    rf"Alert count vs. focal-plane radius — fit: $N(r)=A\,\cos(B\,r)$ on $r\leq{R_FIT_MAX:.0f}$ mm",
    fontsize=11,
)
ax.set_xlim(0, r_max_plot)
ax.set_ylim(bottom=0)
ax.legend(fontsize=7.5, loc="upper right")

savefig("alert_count_vs_radius_cosfit")
plt.show()

In [ ]:
# ── Residuals plot ────────────────────────────────────────────────────────────
# Show (N_observed - N_fit) / sqrt(N_observed) per CCD to diagnose
# CCDs that deviate significantly from the cosine model
# (dead columns, vignetting, DDF field geometry, etc.)
# Only the fit-range CCDs (r <= R_FIT_MAX) are used here.

if fit_ok:
    N_pred = cosine_model(geom_inner["r_mm"].values, A_fit, B_fit)
    pull = (geom_inner["n_alerts"].values - N_pred) / geom_inner["n_err"].values

    fig2, axes2 = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

    # Left: pull vs r  (fit range only)
    axes2[0].scatter(
        geom_inner["r_mm"],
        pull,
        c=pull,
        cmap="RdBu_r",
        vmin=-3,
        vmax=3,
        s=30,
        zorder=3,
    )
    axes2[0].axhline(0, color="k", lw=0.8, ls="--")
    axes2[0].axhline(+2, color="gray", lw=0.6, ls=":")
    axes2[0].axhline(-2, color="gray", lw=0.6, ls=":")
    axes2[0].set_xlabel(r"$r$ [mm]", fontsize=10)
    axes2[0].set_ylabel(
        r"Pull = $(N_{\rm obs} - N_{\rm fit})\,/\,\sqrt{N_{\rm obs}}$",
        fontsize=9,
    )
    axes2[0].set_title(
        rf"Residuals vs. radius  (fit range $r \leq {R_FIT_MAX:.0f}$ mm)",
        fontsize=10,
    )

    # Right: pull histogram
    axes2[1].hist(
        pull,
        bins=20,
        color="steelblue",
        edgecolor="k",
        linewidth=0.5,
    )
    axes2[1].axvline(0, color="k", lw=1)
    axes2[1].axvline(+2, color="gray", lw=0.8, ls=":")
    axes2[1].axvline(-2, color="gray", lw=0.8, ls=":")
    axes2[1].set_xlabel(
        r"Pull = $(N_{\rm obs} - N_{\rm fit})\,/\,\sqrt{N_{\rm obs}}$",
        fontsize=9,
    )
    axes2[1].set_ylabel("Number of CCDs", fontsize=9)
    axes2[1].set_title(
        f"Pull distribution  (mean={np.mean(pull):.2f},  std={np.std(pull):.2f})",
        fontsize=10,
    )

    fig2.suptitle(
        r"Fit residuals: $N(r) = A\,\cos(B\,r)$   —   "
        f"$A={A_fit:.0f}$,  $B={B_fit:.5f}$ rad/mm,  "
        f"$\\chi^2_{{\\rm red}}={chi2_red:.2f}$"
        f"  [fit range $r \\leq {R_FIT_MAX:.0f}$ mm]",
        fontsize=10,
    )

    savefig("alert_count_vs_radius_residuals")
    plt.show()
else:
    print("Fit did not converge — residuals plot skipped.")